# 📐 Notebook 02 — Métriques Clés
**Objectif** : Calculer les KPIs de satisfaction (global, par service, par profil, NPS)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('..')
from src.metrics import rapport_complet, satisfaction_par_service, correlations_satisfaction

sns.set_theme(style='whitegrid')
df = pd.read_csv('../data/processed/retraite_clean.csv')
print(f'✅ Données chargées : {df.shape[0]} résidents')

## 1. Rapport complet des métriques

In [ ]:
resultats = rapport_complet(df)

## 2. Visualisation — Satisfaction par service

In [ ]:
par_service = satisfaction_par_service(df)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['tomato' if v < par_service['Moyenne'].mean() else 'steelblue'
          for v in par_service['Moyenne']]
par_service['Moyenne'].plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(par_service['Moyenne'].mean(), color='black', linestyle='--', linewidth=1.5,
           label=f'Moyenne globale : {par_service["Moyenne"].mean():.2f}')
ax.set_title('Satisfaction moyenne par service (sur 5)', fontsize=13, fontweight='bold')
ax.set_xlabel('Note moyenne')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/satisfaction_par_service.png', dpi=150)
plt.show()
print('Rouge = sous la moyenne | Bleu = au-dessus')

## 3. Visualisation — Satisfaction par type de Public

In [ ]:
par_public = df.groupby('Public')['Satisfaction'].mean().sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
par_public.plot(kind='bar', ax=ax,
                color=['tomato', 'orange', 'steelblue'], edgecolor='white')
ax.set_title('Satisfaction moyenne par type de Public', fontsize=13, fontweight='bold')
ax.set_ylabel('Satisfaction moyenne (sur 10)')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/satisfaction_par_public.png', dpi=150)
plt.show()
print('Interprétation : Les résidents Valides sont les moins satisfaits (confirmé par ANOVA)')

## 4. Corrélations avec la Satisfaction

In [ ]:
corr = correlations_satisfaction(df)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['steelblue' if v > 0 else 'tomato' for v in corr]
corr.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Corrélations de Pearson avec la Satisfaction', fontsize=13, fontweight='bold')
ax.set_xlabel('Coefficient de corrélation')
plt.tight_layout()
plt.savefig('../reports/figures/correlations.png', dpi=150)
plt.show()
print('→ Le prix (PRIX_PENSION) a une corrélation quasi nulle (r ≈ 0.095)')

## 5. NPS Interne

In [ ]:
from src.metrics import nps_interne
nps = nps_interne(df)

labels = ['Promoteurs\n(≥8)', 'Passifs\n(6-7)', 'Détracteurs\n(≤5)']
values = [nps['promoteurs_%'], nps['passifs_%'], nps['detracteurs_%']]
colors = ['#2ecc71', '#f39c12', '#e74c3c']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val}%', ha='center', fontweight='bold')
ax.set_title(f'NPS Interne = {nps["NPS"]}', fontsize=13, fontweight='bold')
ax.set_ylabel('Pourcentage de résidents (%)')
plt.tight_layout()
plt.savefig('../reports/figures/nps.png', dpi=150)
plt.show()

In [ ]:
print('✅ Notebook 02 terminé — passer au notebook 03_visualisations.ipynb')